# Gun 1 - Text Splitter

Paragraf/satir/cumle/kelime sinirini sirayla deneyen, token bazli boyut siniriyla calisan ozel bir recursive splitter.

In [1]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from text_splitter import split_text, count_tokens

print("text_splitter yuklendi.")

text_splitter yuklendi.


## 1. Temel davranis testleri

In [2]:
# chunk_size'dan kucuk metin: tek chunk donmeli, bolunmemeli
short_text = "Merhaba dunya. Bu kisa bir test metni."
result = split_text(short_text, chunk_size=50, chunk_overlap=10)
assert len(result) == 1
assert result[0]["text"] == short_text
print("OK - kisa metin tek chunk olarak donuyor:", result)

OK - kisa metin tek chunk olarak donuyor: [{'chunk_id': 0, 'text': 'Merhaba dunya. Bu kisa bir test metni.', 'token_count': 14}]


In [3]:
# chunk_overlap >= chunk_size gecersiz bir konfigurasyon olmali
try:
    split_text("herhangi bir metin", chunk_size=20, chunk_overlap=20)
    print("HATA: ValueError beklenirken firlatilmadi!")
except ValueError as e:
    print(f"OK - beklenen hata yakalandi: {e}")

OK - beklenen hata yakalandi: chunk_overlap, chunk_size'dan kucuk olmali.


## 2. Gercek belge verisiyle test

Test belgeleri birlestirilip parcalaniyor.

In [4]:
GT_PATH = "../data/processed/ground_truth.json"
with open(GT_PATH, encoding="utf-8") as f:
    ground_truth = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


documents = [format_document(v) for _, v in sorted(ground_truth.items())]
corpus = "\n\n---\n\n".join(documents)

print(f"Corpus toplam token sayisi: {count_tokens(corpus)}")
print()
print(corpus[:400], "...")

Corpus toplam token sayisi: 417

Talep Eden: Dila Alpay
Tarih: 2026-08-12
Departman: Yazılım Geliştirme
Konu: Ek Monitör Talebi

Geliştirmekte olduğum yapay zeka destekli doküman analiz modülünün test işlemleri sırasında, log takibi ve kod yazımını eşzamanlı yürütebilmek amacıyla tarafıma 1 adet 27 inç monitör tahsis edilmesini rica ederim.

---

Talep Eden: Ahmet Yilmaz
Tarih: 2026-08-13
Departman: Insan Kaynaklari
Konu: Yeni La ...


In [5]:
# Corpus kucuk oldugu icin (embedding modellerinin gercek limitlerine kiyasla),
# bolme davranisini gozle gorulur kilmak amaciyla dusuk bir chunk_size kullaniyoruz.
# Gercek kullanimda (Gun 2'de secilecek embedding modeline gore) bu deger
# genelde 200-500 token araliginda olur.
chunks = split_text(corpus, chunk_size=60, chunk_overlap=15)

print(f"{len(chunks)} chunk uretildi.\n")
for c in chunks:
    print(f"--- chunk {c['chunk_id']} ({c['token_count']} token) ---")
    print(c["text"])
    print()

11 chunk uretildi.

--- chunk 0 (60 token) ---
Talep Eden: Dila Alpay
Tarih: 2026-08-12
Departman: Yazılım Geliştirme
Konu: Ek Monitör Talebi

Geliştirmekte olduğum yapay zeka destekli doküman

--- chunk 1 (60 token) ---
olduğum yapay zeka destekli doküman analiz modülünün test işlemleri sırasında, log takibi ve kod yazımını eşzamanlı yürütebilmek amacıyla tarafıma 1

--- chunk 2 (30 token) ---
amacıyla tarafıma 1 adet 27 inç monitör tahsis edilmesini rica ederim.

---

--- chunk 3 (55 token) ---
tahsis edilmesini rica ederim.

---

Talep Eden: Ahmet Yilmaz
Tarih: 2026-08-13
Departman: Insan Kaynaklari
Konu: Yeni Laptop Talebi

--- chunk 4 (61 token) ---
Kaynaklari
Konu: Yeni Laptop Talebi

Mevcut laptopumun performans sorunlari nedeniyle is verimliligimi etkilemektedir. Tarafima yeni bir laptop tahsis edilmesini rica ederim.

--- chunk 5 (55 token) ---
yeni bir laptop tahsis edilmesini rica ederim.

---

Talep Eden: Zeynep Kaya
Tarih: 2026-08-14
Departman: None
Konu: Klavye Degisimi



## 3. Kontrol: hicbir kelime bolunmemis mi?

In [6]:
import re

corpus_words = set(re.findall(r"\w+", corpus, flags=re.UNICODE))

broken = []
for c in chunks:
    for word in re.findall(r"\w+", c["text"], flags=re.UNICODE):
        if word not in corpus_words:
            broken.append((c["chunk_id"], word))

if broken:
    print("UYARI - orijinal corpus'ta olmayan (bolunmus olabilecek) kelimeler:")
    for chunk_id, word in broken:
        print(f"  chunk {chunk_id}: {word!r}")
else:
    print("OK - tum chunk'lardaki kelimeler orijinal corpus ile birebir eslesiyor, kirik kelime yok.")

OK - tum chunk'lardaki kelimeler orijinal corpus ile birebir eslesiyor, kirik kelime yok.
